<!-- # [LAB-03] 지도학습-예측모델ㅣ01-프로젝트 개요(연습문제).ipynb -->

In [106]:
# ## 1. rdiff_flag 값 비교 — 무슨 의미인지

# **계산 원리**: `rel_diff = |평균 - 중앙값| / 중앙값` 이런 식으로 구해서
# - 0.1 미만 → similar (평균과 중앙값이 비슷 = 분포가 대칭에 가까움)
# - 0.1~0.5 → diff (좀 치우침)
# - 0.5 이상 → large_diff (심하게 치우침, 극단값 영향 큼)

# **캘리포니아**: diff 5개, similar 4개 (총 9개 변수)
# **킹카운티**: similar 9개, large_diff 6개, diff 4개 (총 19개 변수)

# 여기서 주의할 점 — **변수 개수 자체가 다름** (캘리포니아는 9개, 킹카운티는 19개). 그래서 단순히 "6개 vs 0개"를 절대 개수로만 비교하면 안 되고, "캘리포니아엔 아예 없던 **등급 자체(large_diff)**가 킹카운티엔 존재한다"는 게 핵심 포인트야. 개수 비교가 아니라 **등급의 유무** 비교.

# ## 2. inf_fields — 왜 이게 중요한지

# 캘리포니아: inf 변수 0개 (Empty DataFrame)
# 킹카운티: inf 변수 4개 (waterfront, view, sqft_basement, yr_renovated)

# 이 4개 변수들은 **중앙값이 0**이야. 그러니까 계산식에서 분모(중앙값)가 0이 되면서 나눗셈이 `inf`로 튀어버린 거지. 이게 왜도(극단적 치우침)가 커서 large_diff가 된 게 **아니라**, 애초에 계산이 불가능해서 예외적으로 large_diff에 섞여 들어간 거야.

# → 그래서 문제 4번("large_diff 중 rel_diff가 유한한 숫자로 계산 안 된 변수 몇 개?")의 답이 **4개**가 나온 거고, 실제로 "값이 진짜 널뛰는" large_diff는 6개 중 2개(sqft_lot, yr_built 등 나머지)뿐이라는 걸 알 수 있어.

# ## 3. 왜도(skew) 비교

# **캘리포니아 상위 5개**: population(4.96), total_rooms(4.16), total_bedrooms(3.46), households(3.41), median_income(1.65)
# → 전부 **구역 단위 집계 변수**들. 인구, 방 개수 총합 같은 건 원래 지역마다 규모 차이가 크니까 왜도가 크게 나오는 게 당연함.

# **킹카운티 상위 5개**: sqft_lot(13.06), waterfront(11.39), sqft_lot15(9.51), yr_renovated(4.55), price(4.02)
# → sqft_lot(대지면적) 왜도가 13.06으로 **캘리포니아 최고치의 2배 이상**. 이건 소수의 초대형 필지(대저택 부지 등)가 전체 분포를 심하게 오른쪽으로 잡아당기고 있다는 뜻.

# ## 종합 해석

# 두 결과를 나란히 놓고 보면 메시지는 이거야:

# > **집계 단위가 바뀌면 (구역 → 개별 주택), 똑같은 "평균-중앙값 거리" 판정 기준이라도 완전히 다른 등급 분포가 나온다.**

# 캘리포니아처럼 이미 여러 주택을 평균 낸 지역 단위 데이터는 극단값이 어느 정도 상쇄돼서 diff 이상으로 안 심해지는데, 킹카운티처럼 주택 1채=1행인 데이터는 극단적으로 크거나 특이한 집 하나가 평균 전체를 왜곡시켜서 large_diff, 초고왜도 변수가 튀어나온다는 거지. 그래서 "캘리포니아에서 검증된 절차"를 그대로 믿으면 안 된다는 팀장의 지적이 실제로 맞았다는 걸 데이터로 증명한 셈이야.

In [107]:
# **"그냥 다른 데이터라서"가 아니라, 그 중에서도 특히 "집계 단위(관측 단위)가 다르기 때문"**이야. 이 둘을 구분하는 게 중요해.

# "다른 데이터"에는 여러 차이가 섞여있음

# 캘리포니아 데이터 vs 킹카운티 데이터는 사실 여러 면에서 다름:

# 지역이 다름 (캘리포니아 전체 vs 시애틀 킹카운티)
# 시점이 다름 (1990년 vs 2014~2015년)
# 관측 단위(집계 단위)가 다름 — 이게 핵심
# 캘리포니아: 1행 = 구역(block group), 그 구역 안 여러 가구를 평균/합산한 값
# 킹카운티: 1행 = 주택 한 채, 개별 관측치 그대로
# 왜 "집계 단위" 차이가 결정적인가

# 지역이 다르거나 시점이 다른 것만으로는 왜도나 large_diff가 이렇게 극적으로 벌어지는 이유를 설명 못 해. 진짜 이유는 통계적인 거야:

# 구역 단위 집계 데이터(캘리포니아)

# 한 행이 "그 구역 내 여러 가구의 평균"이기 때문에, 이미 한 번 평균을 낸 결과
# 중심극한정리처럼, 여러 개체를 평균 내면 극단값(아주 크거나 작은 값)이 서로 상쇄되면서 분포가 완만해짐
# 그래서 원래 개별 가구 단위였으면 있었을 극단치들이 이미 뭉개져서 안 보임

# 개별 관측 단위 데이터(킹카운티)

# 한 행이 실제 주택 한 채이므로, 초대형 저택이나 초대형 필지가 있으면 그 값 하나가 그대로 살아있음
# 평균이 이런 소수의 극단값에 쉽게 끌려감 (평균은 극단값에 민감, 중앙값은 안 그럼)
# 그 결과 평균-중앙값 거리(rel_diff)가 크게 벌어지고, 왜도도 커짐
# 정리

# 즉, "지역이 다르다"거나 "시점이 다르다"는 이 통계량 차이를 직접 설명하지 못해. 하지만 "집계 단위가 구역 평균이냐, 개별 관측치냐"는 이 데이터의 분포 모양 자체를 결정짓는 핵심 요인이라서, 팀장이 지적한 "캘리포니아에서 검증됐다고 이 데이터에도 적용된다는 보장 없다"는 말의 진짜 근거가 바로 이 집계 단위 차이인 거야.

In [108]:
# 캘리포니아

# rdiff_flag: diff 5개, similar 4개 → large_diff는 0개
# inf 변수도 0개 (Empty DataFrame)
# 왜도 상위: population, total_rooms, total_bedrooms, households, median_income (구역 집계 변수들)

# kc_house

# rdiff_flag: similar 9개, large_diff 6개, diff 4개
# inf 변수 4개 (waterfront, view, sqft_basement, yr_renovated)
# 왜도 상위: sqft_lot(13.06)이 압도적

# 이렇게 같은 판정 기준, 같은 코드를 두 데이터셋에 적용했을 때

# 캘리포니아는 가장 심해봤자 diff까지만 나왔는데
# 킹카운티는 large_diff가 6개나 나옴


In [109]:
# 이 주제 관련해서 놓치기 쉬운 것들 몇 개 짚어줄게.

# ## 1. 이건 통계학에서 유명한 문제야 — MAUP / 생태학적 오류
# 지금 겪은 문제는 사실 이름이 있어:
# - **MAUP (Modifiable Areal Unit Problem)**: 같은 데이터라도 어떤 단위로 묶어서 집계하느냐(구역 단위 vs 개별 단위)에 따라 통계 결과(평균, 상관관계, 분산 등)가 달라지는 현상
# - **생태학적 오류 (Ecological Fallacy)**: 집계된 데이터(구역 평균)로 얻은 결론을 개별 단위(개인/개별 주택)에 그대로 적용하면 틀릴 수 있다는 것

# → 팀장이 "1990년 캘리포니아 한 건으로 검증됐다고 할 수 없다"고 한 게 바로 이 개념을 지적한 거야. 리포트에 이 용어를 언급하면 훨씬 탄탄해 보일 거야.

# ## 2. 평균 vs 중앙값 — 왜 이 둘의 거리를 보는지 원리
# - 평균은 모든 값을 더해서 나누니까 극단값 하나에도 쉽게 흔들림
# - 중앙값은 순서상 가운데 값이라 극단값에 거의 영향 안 받음
# - 그래서 "평균-중앙값 거리가 크다" = "극단값(이상치)이 존재하거나 분포가 한쪽으로 심하게 쏠려있다"는 뜻으로 해석하는 거야

# ## 3. 왜도(skewness)랑 이 지표가 사실 같은 얘기를 하고 있다는 것
# rel_diff랑 skew는 계산식은 다르지만 **둘 다 "분포가 대칭인가 아닌가"를 재는 지표**야. 그래서 large_diff로 판정된 변수(sqft_lot 등)가 왜도 상위 리스트에도 같이 나오는 게 우연이 아니라 당연한 결과. 이 둘이 서로 교차검증되고 있다는 걸 리포트에 언급하면 좋아.

# ## 4. inf 처리할 때 코드 로직의 허점
# `waterfront`, `view` 같은 변수는 애초에 0/1 이진값이거나 대부분 0인 변수라서 "평균-중앙값 상대거리"라는 지표 자체가 안 맞는 변수야. 이런 변수엔 이 판정 로직을 아예 적용하면 안 되는 게 맞고, 지금처럼 inf가 나온 걸 "심각한 등급"으로 자동 분류해버리면 오해를 부를 수 있어. → 리포트에서 "이 4개는 계산식 한계로 인한 오분류이지 실제 심각한 치우침이 아니다"라고 명확히 구분해서 써야 해.

# ## 5. log 변환은 왜 하는지 (문제 5번과 연결)
# sqft_lot처럼 왜도가 극심한(오른쪽 꼬리 긴) 변수는 로그 변환(`np.log1p`)하면 분포가 정규분포에 가까워짐. 이건 이후 회귀분석이나 모델링에서 성능에 영향 주니까, 지금 단계에서 "이 변수는 로그 변환 검토 대상"이라고 미리 표시해두는 게 실무에서 흔한 절차야.

In [110]:
# ## 이게 왜 중요한지 — 더 넓은 배경

# ### 1. 이 문제가 통계학 역사에서 얼마나 오래되고 뿌리깊은 문제인지

# MAUP는 1979년 지리학자 Openshaw가 이름 붙였지만, 그 이전부터 사회과학에서 계속 골칫거리였어. **핵심은 이거야: 우리가 "데이터"라고 부르는 것 자체가 이미 어떤 선택(집계 방식)의 결과물**이라는 것. 원자료는 절대 "객관적으로 주어진 것"이 아니라, 누군가 "이 단위로 묶자"고 결정한 순간부터 이미 왜곡이 시작됨. 이게 무서운 이유는, **분석가가 원자료를 받았을 때 그게 이미 가공된 결과라는 걸 인지 못 하면, 자기가 뭘 분석하고 있는지도 모른 채 결론을 내리게 된다**는 거야.

# ### 2. "생태학적 오류" — 실제로 사람이 죽거나 정책이 잘못되는 수준의 사례들

# 이 개념이 그냥 통계 수업용 장난감이 아니라는 걸 보여주는 유명한 사례들이 있어:

# - **로빈슨의 역설 (1950)**: 미국 각 주(state) 단위로 "이민자 비율"과 "문맹률"의 상관관계를 봤더니 **양의 상관**(이민자 많은 주가 문맹률도 높음)이 나왔어. 근데 개인 단위로 보면 실제로는 **이민자 개인이 오히려 문맹률이 낮았음**. 왜냐면 이민자들이 문맹률 낮은 대도시(뉴욕 등)에 몰려 살았기 때문. 주(state) 단위로 집계한 순간 개인 수준의 진짜 관계가 완전히 뒤집혀버린 거지.

# - **레드라이닝/부동산 정책**: 구역 단위(census tract, block group)로 집계된 소득·인종 데이터를 근거로 대출 심사, 보험료, 정책자금 배분을 하는 경우가 많은데, 이 구역 경계를 어떻게 긋느냐에 따라 "이 동네는 위험하다/아니다"는 결론이 바뀔 수 있어. 캘리포니아 데이터도 정확히 이 구조(block group 집계)라서, 지금 배우는 게 실제 부동산·정책 데이터 분석에서 흔히 만나는 함정과 같은 거야.

# ### 3. 데이터 분석가 실무에서 왜 매일 마주치는 문제인지

# 너가 목표로 하는 커머스 플랫폼(무신사, 올리브영 등) 데이터 분석에서도 똑같은 함정이 있어:

# - "일별 매출"로 볼 때와 "주문 건별"로 볼 때 트렌드가 다르게 보임
# - "카테고리 평균 구매액"과 "고객 개인별 구매액"은 완전히 다른 얘기를 함 (고액 구매자 소수가 카테고리 평균을 확 끌어올릴 수 있음)
# - A/B 테스트 결과를 "세션 단위"로 볼지 "유저 단위"로 볼지에 따라 전환율이 다르게 계산됨
# - 지역별 집계 데이터(시/군/구 평균 소득 등)로 개인 소비 패턴을 추정하면 로빈슨 역설처럼 뒤집힐 수 있음

# 즉 "구역 단위 vs 개별 단위"는 부동산 데이터에만 있는 특수 사례가 아니라, **"내가 지금 보는 이 숫자가 어떤 단위로 집계된 건지" 항상 확인해야 한다**는 데이터 분석가의 기본 습관을 만들어주는 훈련이야.

# ### 4. 모델링 단계로 넘어갈 때의 직접적 영향

# 이건 LAB-03이 "지도학습-예측모델" 단원이라는 걸 생각하면 더 중요해:

# - 캘리포니아 데이터로 회귀모델을 만들면, 이미 평균낸 값이라 **잔차(residual)가 실제보다 작아 보이고 모델이 과도하게 잘 맞는 것처럼 착각**할 수 있어 (개별 관측치의 노이즈가 이미 평균으로 지워졌으니까)
# - 킹카운티처럼 개별 단위 데이터는 노이즈와 이상치가 그대로 살아있어서, **모델 성능(R² 등)이 훨씬 낮게 나오는 게 정상**인데, 이 배경 지식이 없으면 "내 모델이 형편없나?"라고 잘못 판단하게 됨
# - sqft_lot처럼 왜도 큰 변수를 로그 변환 없이 그대로 넣으면 선형회귀 가정(정규성, 등분산성)이 깨져서 모델이 왜곡됨

# ### 정리 — 한 줄로 요약하면

# **"숫자 자체보다, 그 숫자가 어떤 단위로 만들어졌는지를 먼저 물어야 한다"**는 게 이 실습이 가르치려는 진짜 교훈이야. 이건 단순 기술이 아니라 **분석가로서의 비판적 사고 습관**이고, 실무 인터뷰에서도 "이 데이터 어떻게 검증했어요?"라는 질문에 "평균/최댓값만 봤어요"가 아니라 "집계 단위부터 확인했어요"라고 답할 수 있게 해주는 근본적인 역량이야.

# [LAB-03] 1. 프로젝트 개요 - 연습문제

## 준비작업

### 라이브러리 참조

In [111]:
from jussam import load_data
from helpers import *
from pandas import DataFrame
import numpy as np

## 📚 캘리포니아에서 끝난 줄 알았던 품질 점검

### 당신은 지난 단원에서 캘리포니아 주택가격 보고서의 Ⅰ장을 마친 분석가입니다.

보고서를 덮으며 팀장이 말합니다. "1990년 캘리포니아 한 건으로 우리 점검 절차가 검증됐다고는 말할 수 없다."

그래서 받은 자료가 2014년 5월부터 2015년 5월까지 시애틀이 속한 킹 카운티에서 실제로 거래된 주택 기록입니다. 캘리포니아가 구역 단위로 집계된 자료였던 것과 달리, 이 자료는 **주택 한 채가 한 행**이고 침실 수·욕실 수·면적·건축 연도처럼 집 한 채의 특성이 그대로 들어 있습니다.

행을 지우거나 값을 고치기 전에, **원본 그대로** 구조·형식·품질을 점검하고 기술통계량까지 확인하세요.

> "캘리포니아에서 쓰던 판정 기준을 그대로 들이대면 이 데이터에서는 무엇이 걸리는지 보고하세요."

### 💻 코드 작성

#### 데이터 가져오기

In [112]:
origin = load_data("kc_house")

# 이진형 변수는 범주형으로 변환 (0/1을 숫자로 두면 연속형 통계에 섞여 들어갑니다.)
origin["waterfront"] = origin["waterfront"].astype("category")

# 우편번호는 숫자로 저장되어 있지만 지역을 가리키는 코드이므로 범주형으로 변환
origin["zipcode"] = origin["zipcode"].astype("category")    

origin.head()

📚 이 데이터 세트는 시애틀이 속한 킹 카운티의 주택 매매 가격 정보를 담고 있습니다. 2014년 5월부터 2015년 5월까지 판매된 주택들이 포함되어 있습니다.

(출처: https://www.kaggle.com/datasets/harlfoxem/housesalesprediction)

    field          description
--  -------------  ------------------------------------------------
 0  date           거래 날짜
 1  price          주택 가격 (USD 달러)
 2  bedrooms       침실 수 (개수)
 3  bathrooms      욕실 수 (개수)
 4  sqft_living    거주 공간 면적 (제곱 피트)
 5  sqft_lot       대지 면적 (제곱 피트)
 6  floors         층 수 (개수)
 7  waterfront     워터프론트 여부 (이진형: 1/0)
 8  view           조망 점수 (0~4)
 9  condition      주택 상태 점수 (1~5)
10  grade          건축 등급 점수 (1~13)
11  sqft_above     지상 면적 (제곱 피트)
12  sqft_basement  지하 면적 (제곱 피트)
13  yr_built       건축 연도
14  yr_renovated   리모델링 연도
15  zipcode        우편번호
16  lat            위도
17  long           경도
18  sqft_living15  15개 인근 주택의 평균 거주 공간 면적 (제곱 피트)
19  sqft_lot15     15개 인근 주택의 평균 대지 면적 (제곱 피트)



,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2014-10-13,221900.000,3,1.000,1180,5650,1.000,0,0,3,7,1180,0,1955,0,98178,47.511,-122.257,1340,5650
1,2014-12-09,538000.000,3,2.250,2570,7242,2.000,0,0,3,7,2170,400,1951,1991,98125,47.721,-122.319,1690,7639
2,2015-02-25,180000.000,2,1.000,770,10000,1.000,0,0,3,6,770,0,1933,0,98028,47.738,-122.233,2720,8062
3,2014-12-09,604000.000,4,3.000,1960,5000,1.000,0,0,5,7,1050,910,1965,0,98136,47.521,-122.393,1360,5000
4,2015-02-18,510000.000,3,2.000,1680,8080,1.000,0,0,3,8,1680,0,1987,0,98074,47.617,-122.045,1800,7503


#### 데이터 구조 점검

In [113]:
print(f"데이터 크기: {origin.shape}")
origin.info()

데이터 크기: (21613, 20)
<class 'pandas.DataFrame'>
RangeIndex: 21613 entries, 0 to 21612
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   date           21613 non-null  datetime64[ns]
 1   price          21613 non-null  float64       
 2   bedrooms       21613 non-null  int64         
 3   bathrooms      21613 non-null  float64       
 4   sqft_living    21613 non-null  int64         
 5   sqft_lot       21613 non-null  int64         
 6   floors         21613 non-null  float64       
 7   waterfront     21613 non-null  category      
 8   view           21613 non-null  int64         
 9   condition      21613 non-null  int64         
 10  grade          21613 non-null  int64         
 11  sqft_above     21613 non-null  int64         
 12  sqft_basement  21613 non-null  int64         
 13  yr_built       21613 non-null  int64         
 14  yr_renovated   21613 non-null  int64         
 15  zipcode   

#### 형식·표기·단위 점검 - 연속형 변수

In [114]:
# 값을 검사할 필드 목록 ( 숫자형 변수 전체)
# --> 범주형으로 바꾼 waterfront, zipcode와 날짜형인 date는 자동으로 빠집니다. 
fields = my_qtcheck.get_number_column_names(origin)

# 위도, 경도는 주택의 위치를 나타내는 좌표이므로 값의 범위를 점검할 대상이 아닙니다.
fields = [f for f in fields if f not in ['lat', 'long']]    
print(f"점검 대상 변수 {len(field)}개: {fields}")

minmax = []  # 최소값과 최대값을 저장할 리스트

for field in fields:
    min_value = origin[field].min()
    max_value = origin[field].max()
    minmax.append({"min": min_value, "max": max_value})

# 최소값과 최대값을 데이터 프레임으로 변환하여 출력
minmax_df = DataFrame(minmax, index=fields)
minmax_df

점검 대상 변수 8개: ['price', 'bedrooms', 'bathrooms', 'sqft_living', 'sqft_lot', 'floors', 'view', 'condition', 'grade', 'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'sqft_living15', 'sqft_lot15']


,min,max
price,75000.000,7700000.000
bedrooms,0.000,33.000
bathrooms,0.000,8.000
sqft_living,290.000,13540.000
sqft_lot,520.000,1651359.000
floors,1.000,3.500
view,0.000,4.000
condition,1.000,5.000
grade,1.000,13.000
sqft_above,290.000,9410.000


In [115]:
# 워터프론트 여부 - 두 값의 분포 확인 
display(origin['waterfront'].value_counts())

# 우편번호 - 지역 수와 가장 거래가 많은 지역 확인 
print(f"우편번호(지역) 수: {origin['zipcode'].unique()}개")
display(origin['zipcode'].value_counts().head())

waterfront
0    21450
1      163
Name: count, dtype: int64

우편번호(지역) 수: [98178, 98125, 98028, 98136, 98074, ..., 98072, 98188, 98014, 98055, 98039]
Length: 70
Categories (70, int64): [98001, 98002, 98003, 98004, ..., 98178, 98188, 98198, 98199]개


zipcode
98103    602
98038    590
98115    583
98052    574
98117    553
Name: count, dtype: int64

#### 데이터 품질 점검 - 중복값과 결측치

In [116]:
df1 = my_qtcheck.check_duplicates(origin)

missing = my_qtcheck.check_missing_values(df1)
print(f"결측치가 있는 변수의 수: {(missing['Missing Count'] > 0).sum()}")

# 0으로 기록된 값의 비율 점검
for field in ['yr_renovated', 'sqft_basement', 'bedrooms']:
    zero_ratio = (df1[field] == 0).mean() * 100
    print(f"{field} 가 0인 비율: {zero_ratio:.1f}% ({(df1[field] == 0).sum()}건)")

중복된 행의 수 : 0
결측치가 있는 변수의 수: 0
yr_renovated 가 0인 비율: 95.8% (20699건)
sqft_basement 가 0인 비율: 60.7% (13126건)
bedrooms 가 0인 비율: 0.1% (13건)


#### 기술 통계량

In [117]:
desc_df = my_qtcheck.numerical_summary(df1)
desc_df[['mean', '50%', 'rel_diff', 'rdiff_flag', 'skew', 'outliers_ratio', 'log_need']]

,mean,50%,rel_diff,rdiff_flag,skew,outliers_ratio,log_need
price,540088.142,450000.000,0.200,diff,4.024,0.053,log
bedrooms,3.371,3.000,0.124,diff,1.974,0.025,log1p
bathrooms,2.115,2.250,0.060,similar,0.511,0.026,log1p
sqft_living,2079.900,1910.000,0.089,similar,1.472,0.026,log
sqft_lot,15106.968,7618.000,0.983,large_diff,13.060,0.112,log
floors,1.494,1.500,0.004,similar,0.616,0.000,none
view,0.234,0.000,inf,large_diff,3.396,0.098,log1p
condition,3.409,3.000,0.136,diff,1.033,0.001,log
grade,7.657,7.000,0.094,similar,0.771,0.088,log
sqft_above,1788.391,1560.000,0.146,diff,1.447,0.028,log


#### 판정 결과 집계

In [118]:
# 판정 등급별 변수 개수
print("[ rdiff_flag 판정별 변수 개수 ]")
print(desc_df['rdiff_flag'].value_counts())

# 상대 거리가 유한한 숫자로 계산되지 않은 변수
inf_fields = desc_df[np.isinf(desc_df['rel_diff'])]
print(f"\n[ rel_diff 가 무한대인 변수: {len(inf_fields)}개 ]")
print(inf_fields[['mean', '50%', 'rel_diff', 'rdiff_flag']])

# 왜도 상위 5개
print("\n[ 왜도 상위 5개 ]")
print(desc_df['skew'].sort_values(ascending=False).head(5))

[ rdiff_flag 판정별 변수 개수 ]
rdiff_flag
similar       8
large_diff    5
diff          4
Name: count, dtype: int64

[ rel_diff 가 무한대인 변수: 3개 ]
                 mean   50%  rel_diff  rdiff_flag
view            0.234 0.000       inf  large_diff
sqft_basement 291.509 0.000       inf  large_diff
yr_renovated   84.402 0.000       inf  large_diff

[ 왜도 상위 5개 ]
sqft_lot       13.060
sqft_lot15      9.507
yr_renovated    4.549
price           4.024
view            3.396
Name: skew, dtype: float64


### 문제 풀이

#### 1. 값의 범위를 점검하던 중, 주택 한 채의 침실 수로는 도저히 성립하지 않는 기록이 눈에 띄었습니다. 이 데이터에 기록된 침실 수의 최댓값은 얼마인가요? (정수)

- **정답**: `33`
- **복습 개념**: 형식·표기·단위 점검입니다. 값이 물리적으로 가능한 범위 안에 있는지를 최솟값·최댓값으로 훑어보는 단계이며, 캘리포니아에서 상한 절단의 단서를 처음 발견했던 바로 그 단계입니다.
- **풀이 접근**: 연속형 변수들을 목록으로 묶어 각각의 최솟값과 최댓값을 구한 뒤 한 표로 모아 봅니다. 그리고 그 표를 변수의 의미와 하나씩 대조합니다. "이 값이 현실에서 가능한가"를 묻는 것이 이 단계의 전부입니다.
- **근거(계산 결과)**: 침실 수의 범위는 **0 ~ 33**입니다. 최댓값 33이 기록된 주택을 열어 보면 거주 면적이 1,620제곱피트(약 45평), 욕실은 1.75개입니다. 45평에 침실 33개는 성립하지 않으므로 **3을 33으로 잘못 입력한 기록**으로 보는 것이 타당합니다.
- **함께 생각해 볼 점**: 같은 표에서 침실 수와 욕실 수의 최솟값이 둘 다 0이라는 점도 눈에 띕니다. 주거용 주택에 침실도 욕실도 없다는 뜻이 되니 이것 역시 정상적인 값은 아닙니다. 다만 이 단계에서는 **발견하고 기록만** 합니다. 고치거나 지우는 것은 뒤의 전처리 단계 일입니다.

#### 2. 결측치를 점검했더니 한 건도 없었습니다. 그런데 리모델링 연도가 0으로 기록된 주택이 있습니다. 이런 주택은 전체의 몇 %인가요? (소수 첫째 자리, % 단위)

- **정답**: `95.8`
- **복습 개념**: 0으로 기록된 값의 해석입니다. 결측치 점검표는 비어 있는 칸만 세기 때문에, **"없음"을 0으로 적어 둔 값은 결측으로 잡히지 않습니다.**
- **풀이 접근**: 먼저 결측치 점검표로 비어 있는 칸이 정말 없는지 확인합니다. 그다음 0이 의미를 갖는 변수를 골라 0인 행의 비율을 직접 계산합니다.
- **근거(계산 결과)**: 리모델링 연도가 0인 주택은 20,699건으로 전체 21,613건의 **95.8%** 입니다. 이 0은 "연도를 모른다"가 아니라 **"리모델링한 적이 없다"** 는 뜻입니다. 같은 방식으로 지하 면적이 0인 주택도 60.7%인데, 이것도 결측이 아니라 지하실이 없다는 뜻입니다.
- **자주 하는 실수**: 0을 결측으로 보고 평균으로 대체하면, 리모델링한 적 없는 집에 "1971년쯤 리모델링했다"는 없던 사실이 생겨납니다. 반대로 0을 그대로 두고 평균을 내면 리모델링 연도의 평균이 84년이라는 이상한 숫자가 나옵니다. 이 변수는 연도가 아니라 **리모델링 여부(0/1)** 로 바꿔 쓰는 편이 맞습니다.

#### 3. 평균과 중앙값의 상대적 거리로 자동 판정된 등급 중, 캘리포니아 데이터에서는 해당 변수가 하나도 없었던 가장 심한 등급이 이 데이터에서는 여러 변수에 붙었습니다. 그 등급의 이름은 무엇인가요? (영문 판정값 1개)

- **정답**: `large_diff`
- **복습 개념**: 평균과 중앙값의 상대적 거리에 따른 판정입니다. 거리가 0.1 미만이면 `similar`, 0.1 이상 0.5 미만이면 `diff`, 0.5 이상이면 `large_diff`로 판정됩니다.
- **풀이 접근**: 기술통계량 결과표에서 판정 열만 뽑아 값별로 개수를 세어 봅니다. 캘리포니아에서는 이 열에 `similar`와 `diff`만 있었다는 점을 떠올리며 비교합니다.
- **근거(계산 결과)**: 판정 결과는 `similar` 9개, `large_diff` 6개, `diff` 4개입니다. 캘리포니아에서는 `large_diff`가 **0개**여서 "평균이 완전히 무너진 변수는 없다"고 결론지었는데, 이 데이터에서는 **6개**나 나왔습니다. 대지 면적(sqft_lot)의 상대 거리는 **0.983** 으로, 평균이 중앙값의 거의 두 배입니다.
- **헷갈리기 쉬운 점**: `large_diff`가 "데이터가 잘못됐다"는 뜻은 아닙니다. 캘리포니아는 구역 단위로 집계된 자료라 값이 어느 정도 평탄해졌지만, 이 자료는 주택 한 채가 한 행이라 소수의 대저택이 평균을 그대로 끌어올립니다. **집계 단위가 다르면 같은 지표도 다르게 나옵니다.**

#### 4. 그 가장 심한 등급을 받은 변수들을 자세히 보면, 일부는 상대적 거리 값 자체가 유한한 숫자로 계산되지 않습니다. 그런 변수는 모두 몇 개인가요? (정수)

- **정답**: `4`
- **복습 개념**: 지표가 성립하지 않는 경우를 알아보는 문제입니다. 상대적 거리는 평균과 중앙값의 차이를 **중앙값으로 나눠** 구하므로, 중앙값이 0이면 0으로 나누게 되어 값이 무한대가 됩니다.
- **풀이 접근**: 기술통계량 결과표에서 상대 거리 열이 유한한 숫자인지 확인하고, 무한대인 행만 걸러 세어 봅니다. 그 행들의 중앙값 열을 함께 보면 원인이 바로 보입니다.
- **근거(계산 결과)**: 무한대로 계산된 변수는 **4개** 입니다 — 워터프론트 여부(waterfront), 조망 점수(view), 지하 면적(sqft_basement), 리모델링 연도(yr_renovated). 네 변수 모두 **중앙값이 0** 입니다. 절반 넘는 주택이 0이라서 중앙값이 0이 되고, 그 결과 나눗셈이 성립하지 않습니다.
- **실무 포인트**: 판정 열에는 `large_diff`라고 찍혀 있지만 이 네 변수는 **판정 자체가 의미 없는 경우**입니다. 결과표의 판정값을 그대로 옮겨 적기 전에 그 값이 어떻게 계산됐는지 확인해야 하는 이유가 여기 있습니다. 애초에 이 네 변수는 0과 1로 나뉘는 성격이라 평균·중앙값으로 중심을 논할 대상이 아닙니다.

#### 5. 팀장에게 "이 변수부터 로그 변환을 검토해야 한다"고 보고하려 합니다. 우측 꼬리가 가장 극심한 변수는 무엇인가요? (변수명 1개)

- **정답**: `sqft_lot`
- **복습 개념**: 왜도(skew)로 분포의 꼬리 방향과 정도를 판정하는 단계입니다. 왜도가 0.5보다 크면 우측 꼬리로 판정하며, 값이 클수록 꼬리가 깁니다.
- **풀이 접근**: 기술통계량 결과표의 왜도 열을 내림차순으로 정렬해 가장 큰 값을 찾습니다. 상대적 거리 판정과 로그 변환 필요 여부 열도 함께 보면 근거가 겹쳐 쌓입니다.
- **근거(계산 결과)**: 대지 면적(**sqft_lot**)의 왜도가 **13.060** 으로 1위입니다. 2위는 워터프론트 여부 11.385지만 이것은 0과 1뿐인 값이라 꼬리를 논할 대상이 아니고, 실질적인 2위는 인근 대지 면적 9.507입니다. sqft_lot은 상대 거리 0.983으로 `large_diff`이고 이상치 비율도 11.2%로 가장 높아, 세 지표가 모두 같은 곳을 가리킵니다.
- **결론**: 팀장에게 드릴 답은 이렇게 정리됩니다. **캘리포니아에서는 상한 절단이 문제였지만 이 데이터에는 절단이 없고, 대신 주택 한 채 단위라서 꼬리가 훨씬 깁니다.** 그래서 걸리는 것은 ① 침실 33개 같은 입력 오류, ② 0을 결측으로 오해할 위험, ③ 면적 계열의 극단적인 우측 꼬리 세 가지입니다. 판정 기준은 그대로 쓸 수 있었지만, **기준이 가리키는 문제는 데이터마다 달랐습니다.**

<!-- # [LAB-03] 1. 프로젝트 개요 - 연습문제

## 준비작업

### 라이브러리 참조 -->

<!-- #### 2. 결측치를 점검했더니 한 건도 없었습니다. 그런데 리모델링 연도가 0으로 기록된 주택이 있습니다. 이런 주택은 전체의 몇 %인가요? (소수 첫째 자리, % 단위)

- **정답**: `95.8`
- **복습 개념**: 0으로 기록된 값의 해석입니다. 결측치 점검표는 비어 있는 칸만 세기 때문에, **"없음"을 0으로 적어 둔 값은 결측으로 잡히지 않습니다.**
- **풀이 접근**: 먼저 결측치 점검표로 비어 있는 칸이 정말 없는지 확인합니다. 그다음 0이 의미를 갖는 변수를 골라 0인 행의 비율을 직접 계산합니다.
- **근거(계산 결과)**: 리모델링 연도가 0인 주택은 20,699건으로 전체 21,613건의 **95.8%** 입니다. 이 0은 "연도를 모른다"가 아니라 **"리모델링한 적이 없다"** 는 뜻입니다. 같은 방식으로 지하 면적이 0인 주택도 60.7%인데, 이것도 결측이 아니라 지하실이 없다는 뜻입니다.
- **자주 하는 실수**: 0을 결측으로 보고 평균으로 대체하면, 리모델링한 적 없는 집에 "1971년쯤 리모델링했다"는 없던 사실이 생겨납니다. 반대로 0을 그대로 두고 평균을 내면 리모델링 연도의 평균이 84년이라는 이상한 숫자가 나옵니다. 이 변수는 연도가 아니라 **리모델링 여부(0/1)** 로 바꿔 쓰는 편이 맞습니다. -->

<!-- #### 3. 평균과 중앙값의 상대적 거리로 자동 판정된 등급 중, 캘리포니아 데이터에서는 해당 변수가 하나도 없었던 가장 심한 등급이 이 데이터에서는 여러 변수에 붙었습니다. 그 등급의 이름은 무엇인가요? (영문 판정값 1개)

- **정답**: `large_diff`
- **복습 개념**: 평균과 중앙값의 상대적 거리에 따른 판정입니다. 거리가 0.1 미만이면 `similar`, 0.1 이상 0.5 미만이면 `diff`, 0.5 이상이면 `large_diff`로 판정됩니다.
- **풀이 접근**: 기술통계량 결과표에서 판정 열만 뽑아 값별로 개수를 세어 봅니다. 캘리포니아에서는 이 열에 `similar`와 `diff`만 있었다는 점을 떠올리며 비교합니다.
- **근거(계산 결과)**: 판정 결과는 `similar` 9개, `large_diff` 6개, `diff` 4개입니다. 캘리포니아에서는 `large_diff`가 **0개**여서 "평균이 완전히 무너진 변수는 없다"고 결론지었는데, 이 데이터에서는 **6개**나 나왔습니다. 대지 면적(sqft_lot)의 상대 거리는 **0.983** 으로, 평균이 중앙값의 거의 두 배입니다.
- **헷갈리기 쉬운 점**: `large_diff`가 "데이터가 잘못됐다"는 뜻은 아닙니다. 캘리포니아는 구역 단위로 집계된 자료라 값이 어느 정도 평탄해졌지만, 이 자료는 주택 한 채가 한 행이라 소수의 대저택이 평균을 그대로 끌어올립니다. **집계 단위가 다르면 같은 지표도 다르게 나옵니다.** -->

<!-- #### 4. 그 가장 심한 등급을 받은 변수들을 자세히 보면, 일부는 상대적 거리 값 자체가 유한한 숫자로 계산되지 않습니다. 그런 변수는 모두 몇 개인가요? (정수)

- **정답**: `4`
- **복습 개념**: 지표가 성립하지 않는 경우를 알아보는 문제입니다. 상대적 거리는 평균과 중앙값의 차이를 **중앙값으로 나눠** 구하므로, 중앙값이 0이면 0으로 나누게 되어 값이 무한대가 됩니다.
- **풀이 접근**: 기술통계량 결과표에서 상대 거리 열이 유한한 숫자인지 확인하고, 무한대인 행만 걸러 세어 봅니다. 그 행들의 중앙값 열을 함께 보면 원인이 바로 보입니다.
- **근거(계산 결과)**: 무한대로 계산된 변수는 **4개** 입니다 — 워터프론트 여부(waterfront), 조망 점수(view), 지하 면적(sqft_basement), 리모델링 연도(yr_renovated). 네 변수 모두 **중앙값이 0** 입니다. 절반 넘는 주택이 0이라서 중앙값이 0이 되고, 그 결과 나눗셈이 성립하지 않습니다.
- **실무 포인트**: 판정 열에는 `large_diff`라고 찍혀 있지만 이 네 변수는 **판정 자체가 의미 없는 경우**입니다. 결과표의 판정값을 그대로 옮겨 적기 전에 그 값이 어떻게 계산됐는지 확인해야 하는 이유가 여기 있습니다. 애초에 이 네 변수는 0과 1로 나뉘는 성격이라 평균·중앙값으로 중심을 논할 대상이 아닙니다. -->

<!-- #### 5. 팀장에게 "이 변수부터 로그 변환을 검토해야 한다"고 보고하려 합니다. 우측 꼬리가 가장 극심한 변수는 무엇인가요? (변수명 1개)

- **정답**: `sqft_lot`
- **복습 개념**: 왜도(skew)로 분포의 꼬리 방향과 정도를 판정하는 단계입니다. 왜도가 0.5보다 크면 우측 꼬리로 판정하며, 값이 클수록 꼬리가 깁니다.
- **풀이 접근**: 기술통계량 결과표의 왜도 열을 내림차순으로 정렬해 가장 큰 값을 찾습니다. 상대적 거리 판정과 로그 변환 필요 여부 열도 함께 보면 근거가 겹쳐 쌓입니다.
- **근거(계산 결과)**: 대지 면적(**sqft_lot**)의 왜도가 **13.060** 으로 1위입니다. 2위는 워터프론트 여부 11.385지만 이것은 0과 1뿐인 값이라 꼬리를 논할 대상이 아니고, 실질적인 2위는 인근 대지 면적 9.507입니다. sqft_lot은 상대 거리 0.983으로 `large_diff`이고 이상치 비율도 11.2%로 가장 높아, 세 지표가 모두 같은 곳을 가리킵니다.
- **결론**: 팀장에게 드릴 답은 이렇게 정리됩니다. **캘리포니아에서는 상한 절단이 문제였지만 이 데이터에는 절단이 없고, 대신 주택 한 채 단위라서 꼬리가 훨씬 깁니다.** 그래서 걸리는 것은 ① 침실 33개 같은 입력 오류, ② 0을 결측으로 오해할 위험, ③ 면적 계열의 극단적인 우측 꼬리 세 가지입니다. 판정 기준은 그대로 쓸 수 있었지만, **기준이 가리키는 문제는 데이터마다 달랐습니다.** -->